In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import shap
import os
import plotly.graph_objects as go
from shap.plots import waterfall, beeswarm
from shap import Explanation, KernelExplainer
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as mcolors
from matplotlib import cm
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, r2_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
### Generic policy ###
if feature == "project_feature" and cate == "Generic policy":
    model = RandomForestClassifier(
        max_features=1, min_samples_leaf=6, min_samples_split=7,
        n_estimators=512, n_jobs=1, random_state=1, warm_start=True
    )
    model_type = "tree"
elif feature == "security_practice" and cate == "Generic policy":
    model = RandomForestClassifier(
        bootstrap=False, max_features=1, min_samples_leaf=3,
        min_samples_split=14, n_estimators=512, n_jobs=1,
        random_state=1, warm_start=True
    )
    model_type = "tree"
elif feature == "project_quality" and cate == "Generic policy":
    model = AdaBoostClassifier(algorithm='SAMME',
                   estimator=DecisionTreeClassifier(max_depth=10),
                   learning_rate=0.010381491760996881, n_estimators=362,
                   random_state=1)
    model_type = "ada"
### Reporting mechanism ###
elif feature == "project_feature" and cate == "Reporting mechanism":
    model = ExtraTreesClassifier(bootstrap=True, criterion='entropy', max_features=1,
                     min_samples_split=7, n_estimators=512, n_jobs=1,
                     random_state=1, warm_start=True)
    model_type = "tree"
elif feature == "security_practice" and cate == "Reporting mechanism":
    model = HistGradientBoostingClassifier(early_stopping=True,
                               l2_regularization=8.908183652101429e-05,
                               learning_rate=0.19911994270380215,
                               max_iter=512, max_leaf_nodes=955,
                               min_samples_leaf=33, n_iter_no_change=2,
                               random_state=1, validation_fraction=None,
                               warm_start=True)
    model_type = "hist"
elif feature == "project_quality" and cate == "Reporting mechanism":
    model = KNeighborsClassifier(n_neighbors=2, p=1, weights='distance') 
    model_type = "kernel"
### Scope of practice ###
elif feature == "project_feature" and cate == "Scope of practice":
    model = RandomForestClassifier(max_features=15, min_samples_leaf=5,
                       min_samples_split=20, n_estimators=512, n_jobs=1,
                       random_state=1, warm_start=True)
    model_type = "tree"
elif feature == "security_practice" and cate == "Scope of practice":
    model = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=9),
                   learning_rate=1.9701455892241493, n_estimators=101,
                   random_state=1)
    model_type = "ada"
elif feature == "project_quality" and cate == "Scope of practice":
    model = RandomForestClassifier(bootstrap=False, criterion='entropy', max_features=4,
                       min_samples_leaf=7, n_estimators=512, n_jobs=1,
                       random_state=1, warm_start=True)
    model_type = "tree"
### User guideline ###
elif feature == "project_feature" and cate == "User guideline":
    model = ExtraTreesClassifier(criterion='entropy', max_features=2, n_estimators=512,
                     n_jobs=1, random_state=1, warm_start=True)
    model_type = "tree"
elif feature == "security_practice" and cate == "User guideline":
    model = HistGradientBoostingClassifier(early_stopping=True,
                               l2_regularization=0.1144885415414585,
                               learning_rate=0.35651231429733377, 
                               max_iter=128, max_leaf_nodes=570,
                               min_samples_leaf=52, n_iter_no_change=20,
                               random_state=1,
                               validation_fraction=0.26745137407982933,
                               warm_start=True)
    model_type = "hist"
elif feature == "project_quality" and cate == "User guideline":
    model = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                   learning_rate=0.16884660398877008, n_estimators=144,
                   random_state=1)
    model_type = "ada"
else:
    raise ValueError("Invalid feature or category selection.")


TreeExplainer 

In [ ]:
if model_type == "tree":
    #### Tree explainer ####
    explainer = shap.TreeExplainer(model)
    shap_values_tree = explainer.shap_values(X_test)

    #### TMean and Absolute ####
    shap_df = pd.DataFrame(shap_values_tree[:, :, 1], columns=X_test.columns)

    # Compute and print raw mean SHAP values
    abs_mean_shap_raw = shap_df.abs().mean(0).sort_values(ascending=False)
    print("Absolute Mean SHAP raw:")
    print(abs_mean_shap_raw)
    # Compute and print raw mean SHAP values
    mean_shap_raw = shap_df.mean(0).sort_values(ascending=False)

    #### prepare to plot ####
    feature_names = X_test.columns.tolist() # Get feature names from X_test

    num_samples, num_features, num_classes = shap_values_tree.shape  
    reshaped_values = shap_values_tree.reshape(num_samples, num_features * num_classes)

    columns = [f"{feature_names[i]}_Class_{j}" for i in range(num_features) for j in range(num_classes)] 

    df = pd.DataFrame(reshaped_values, columns=columns)

    # Select only columns corresponding to Class 1
    class_1_columns = [col for col in columns if "_Class_1" in col]  
    df_class_1 = df[class_1_columns]  

    df_class_1.columns = [col.replace("_Class_1", "") for col in df_class_1.columns] 

    #### Plot ####
    # Compute absolute mean SHAP values for correct sorting
    df_mean = df_class_1.abs().mean(numeric_only=True).reset_index() 
    df_mean.columns = ['Feature', 'Mean |SHAP Value|']
    df_mean = df_mean.sort_values(by="Mean |SHAP Value|", ascending=True) 

    # Compute actual mean SHAP values for correct bar coloring
    df_signed_mean = df_class_1.mean(numeric_only=True).reset_index()
    df_signed_mean.columns = ['Feature', 'Mean SHAP Value']

    # Merge both to get proper ordering and sign information
    df_final = df_mean.merge(df_signed_mean, on="Feature")

    # Assign colors: red for positive, blue for negative values
    df_final["Color"] = df_final["Mean SHAP Value"].apply(lambda x: "mediumturquoise" if x >= 0 else "crimson")

    # Creating the horizontal bar plot with correct order
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=df_final["Mean |SHAP Value|"],
        y=df_final["Feature"],
        orientation='h',
        marker=dict(color=df_final["Color"]),
        text=[f"{v:.3f}" for v in df_final["Mean |SHAP Value|"]], 
        textposition="outside"
    ))

    # Updating layout to match SHAP importance plot style
    fig.update_layout(

        title=f"{cate}:{feature}",
        title_font_size=24,
        xaxis_title="Mean SHAP Value (Average Impact on Model Output)",
        xaxis_title_font_size=20, 
        # yaxis_title="Features",
        # yaxis_title_font_size=21,
        template="plotly_white",
        font=dict(size=18), 
        xaxis=dict(
            range=model_range,
            tickfont=dict(size=18),  
            zeroline=True,  
            zerolinecolor="black",
            zerolinewidth=2,
        ),
        yaxis=dict(
            tickfont=dict(size=18) 
        ),
        showlegend=False, 
        width=1200,  
        height=500,  
    )

    # Show the plot
    fig.show()


HistGradientBoost

In [ ]:
if model_type == "hist":
    #### Tree explainer ####
    explainer = shap.TreeExplainer(model)
    shap_values_tree = explainer.shap_values(X_test)

    # Check if SHAP values are a list 
    if isinstance(shap_values_tree, list):  
        shap_values_tree = shap_values_tree[1]  

    # Ensure it's a NumPy array
    shap_values_tree = np.array(shap_values_tree)

    #### Mean and Absolute Mean SHAP values ####
    shap_df = pd.DataFrame(shap_values_tree, columns=X_test.columns)

    # Compute absolute mean SHAP values
    abs_mean_shap_raw = shap_df.abs().mean(0).sort_values(ascending=False)

    # Compute actual mean SHAP values
    mean_shap_raw = shap_df.mean(0).sort_values(ascending=False)

    #### Prepare Data for Plot ####
    feature_names = X_test.columns.tolist()  # Get feature names from X_test

    # Convert SHAP values to DataFrame
    df_class_1 = pd.DataFrame(shap_values_tree, columns=feature_names)

    #### Plot ####
    # Compute absolute mean SHAP values for correct sorting
    df_mean = df_class_1.abs().mean().reset_index()
    df_mean.columns = ['Feature', 'Mean |SHAP Value|']
    df_mean = df_mean.sort_values(by="Mean |SHAP Value|", ascending=True)  

    # Compute actual mean SHAP values for correct bar coloring
    df_signed_mean = df_class_1.mean(numeric_only=True).reset_index()
    df_signed_mean.columns = ['Feature', 'Mean SHAP Value']

    # Merge both to get proper ordering and sign information
    df_final = df_mean.merge(df_signed_mean, on="Feature")

    # Assign colors: red for positive, blue for negative values
    df_final["Color"] = df_final["Mean SHAP Value"].apply(lambda x: "mediumturquoise" if x >= 0 else "crimson")

    # Creating the horizontal bar plot with correct order
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=df_final["Mean |SHAP Value|"],
        y=df_final["Feature"],
        orientation='h',
        marker=dict(color=df_final["Color"]),
        text=[f"{v:.3f}" for v in df_final["Mean |SHAP Value|"]],  
        textposition="outside"
    ))

    # Updating layout to match SHAP importance plot style
    fig.update_layout(
    title=f"{cate}:{feature}",
    title_font_size=24,
    xaxis_title="Mean SHAP Value (Average Impact on Model Output)",
    xaxis_title_font_size=20,
    yaxis_title="Features",
    yaxis_title_font_size=21,
    template="plotly_white",
    font=dict(size=18),
    xaxis=dict(
        range=model_range, 
        tickfont=dict(size=18),
        zeroline=True,
        zerolinecolor="black",
        zerolinewidth=2,
        automargin=False,
    ),
    yaxis=dict(
        tickfont=dict(size=18),
        automargin=True, 
    ),
    showlegend=False,
    width=1400, 
    height=600, 
    margin=dict(l=400, r=50, t=50, b=50),  
    )

    # Show the plot
    fig.show()
    

KernelExplainer 

In [ ]:
if model_type == "kernel" or model_type == "ada":
    #### Kernel Explainer ####
    background = X_train.sample(n=50, random_state=42)  
    explainer = shap.KernelExplainer(model.predict, background)
    shap_values_kernel = explainer.shap_values(X_test, nsamples=100)

    # Ensure SHAP values are a NumPy array
    shap_values_kernel = np.array(shap_values_kernel)

    #### Mean and Absolute Mean SHAP values ####
    shap_df = pd.DataFrame(shap_values_kernel, columns=X_test.columns)

    # Compute absolute mean SHAP values
    abs_mean_shap_raw = shap_df.abs().mean(0).sort_values(ascending=False)
    print("Absolute Mean SHAP raw:")
    print(abs_mean_shap_raw)

    # Compute actual mean SHAP values
    mean_shap_raw = shap_df.mean(0).sort_values(ascending=False)

    #### Prepare Data for Plot ####
    feature_names = X_test.columns.tolist() 
    df_class_1 = pd.DataFrame(shap_values_kernel, columns=feature_names)

    #### Plot ####
# Calculate the % of absolute SHAP values to use as a denominator for percentage calculations
total_shap_sum = df_final['Adjusted SHAP Value'].abs().sum()

# Calculate percentage of each SHAP value relative to the total
df_final['Percentage SHAP Value'] = (df_final['Adjusted SHAP Value'] / total_shap_sum) 

# Print the updated dataframe to see the percentage values
print(df_final[['Feature', 'Percentage SHAP Value']])

# Creating the horizontal bar plot for percentage SHAP values
fig_percentage = go.Figure()
fig_percentage.add_trace(go.Bar(
    x=df_final["Percentage SHAP Value"],
    y=df_final["Feature"],
    orientation='h',
    marker=dict(color=df_final["Color"]), 
    text=[f"{v:.2f}" for v in df_final["Percentage SHAP Value"]],  
    textposition="outside",
    textfont=dict(size=34.5, color='black'),
))

# Updating layout to match SHAP importance plot style, adjusted for percentage values
fig_percentage.update_layout(
    title=f"{cate} : {feature}",  
    title_font_size=48,
    title_font=dict(color='black'),
    xaxis_title="Weight of SHAP Value",
    xaxis_title_font_size=39,
    template="plotly_white",
    xaxis=dict(
        range=[-1.1, 1.1],
        tickfont=dict(size=38),
        zeroline=True,
        zerolinecolor="black",
        zerolinewidth=2,
        automargin=False,
        color='black'
    ),
    yaxis=dict(
        tickfont=dict(size=38),
        automargin=True,
        color='black'

    ),
    showlegend=False,
    width=1420,
    height=650,
    margin=dict(l=450, r=30, t=70, b=120),
)

# Show the plot
fig_percentage.show()
